# **Word2Vec: обучение и анализ эмбеддингов**

***Джон Френч - Солнечная война (фантастический роман)***


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
import sys
print(f"Python: {sys.version}")
print(sys.executable)

import numpy as np
print(f"numpy: {np.__version__}")

import sklearn
print(f"sklearn: {sklearn.__version__}")

import nltk
print(f"nltk: {nltk.__version__}")

import gensim
print(f"gensim: {gensim.__version__}")

import matplotlib
print(f"matplotlib: {matplotlib.__version__}")

import seaborn as sns
print(f"seaborn: {sns.__version__}")

import pandas as pd
print(f"pandas: {pd.__version__}")

import torch
print(f"torch: {torch.__version__}")

print(f"np.int существует: {hasattr(np, 'int')}")

print("\nВсе библиотеки на месте!")

In [ ]:
# Первый запуск - скачиваем данные для nltk.
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')  # Может понадобиться для новой версиию

Подготовка данных и предобработка

In [ ]:
import nltk
import re
import numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Функция чтения корпуса из txt файла.
def read_corpus(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        # Читаем весь файл и разбиваем по строкам.
        # Для книги лучше разбивать по абзацам или предложениям.
        text = f.read()
        # Разбиваем по точкам, вопросительным и восклицательным знакам.
        import re
        sentences = re.split(r'[.!?]+', text)
        return [s.strip() for s in sentences if len(s.strip()) > 20]

# Укажите путь к вашему файлу.
corpus = read_corpus('Джон Френч_Солнечная война.txt')
print(f'Всего предложений в корпусе: {len(corpus)}')

# Стоп-слова для русского языка.
stop_words = set(stopwords.words('russian'))

def clean_and_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # убираем пунктуацию
    text = re.sub(r'\d+', '', text) # убираем цифры
    tokens = word_tokenize(text, language='russian')
    # Удаляем стоп-слова и слишком короткие токены.
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Предобработка всего корпуса.
sentences = [clean_and_tokenize(doc) for doc in corpus if doc.strip()]
sentences = [s for s in sentences if len(s) > 0]
print(f'Предложений после очистки: {len(sentences)}')

# Покажем пример.
if len(sentences) > 0:
    print(f'Пример токенов: {sentences[0][:15]}')

**Задания 1-3**

Обучение базовой модели *(Задания 1–3)*

In [ ]:
# Модель по умолчанию: вектор 100, окно 5, min_count=2.
model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)
model.save("word2vec_book.model")
print("Модель обучена и сохранена")

# Задание 1: похожие слова.
word = "война"
if word in model.wv:
    similar = model.wv.most_similar(word, topn=5)
    print(f"5 слов, похожих на '{word}':")
    for w, score in similar:
        print(f"  {w}: {score:.4f}")
else:
    print(f"Слово '{word}' отсутствует в словаре модели. Попробуйте другое слово.")
    print(f"Доступные слова (первые 20): {list(model.wv.index_to_key)[:20]}")

# Задание 2: аналогии
def analogy(positive, negative, model, topn=1):
    try:
        res = model.wv.most_similar(positive=positive, negative=negative, topn=topn)
        return res[0][0] if topn == 1 else res
    except:
        return "не найдено"

print("\nкороль - мужчина + женщина ≈", analogy(['король', 'женщина'], ['мужчина'], model))
print("солнце - небо + море ≈", analogy(['солнце', 'море'], ['небо'], model))

print("Paris - France + Germany ≈", analogy(['paris', 'germany'], ['france'], model))
print("good - bad + happy ≈", analogy(['good', 'happy'], ['bad'], model))

# Задание 3: косинусное сходство
def cosine_sim(word1, word2, model):
    try:
        return model.wv.similarity(word1, word2)
    except KeyError:
        return None

print("\nКосинусное сходство:")
print("  близкие слова (если есть):", cosine_sim("человек", "люди", model))
print("  случайные слова:", cosine_sim("солнце", "стол", model))

**Задание 4**

Влияние гиперпараметров *(Задание 4)*

In [ ]:
params = [
    {'window': 2, 'size': 50},
    {'window': 5, 'size': 100},
    {'window': 10, 'size': 300},
]

models_params = {}
for p in params:
    key = f"w{p['window']}_s{p['size']}"
    models_params[key] = Word2Vec(sentences, vector_size=p['size'], window=p['window'], 
                                  min_count=2, workers=4)
    print(f"Обучил {key}")

# Тест аналогии для сравнения
positive = ['король', 'женщина']
negative = ['мужчина']
print("\nРезультаты аналогии король - мужчина + женщина:")
for key, m in models_params.items():
    try:
        res = m.wv.most_similar(positive=positive, negative=negative, topn=1)[0][0]
        print(f"  {key}: ≈ {res}")
    except:
        print(f"  {key}: ошибка")

**Задание 5**

CBOW vs Skip-gram *(Задание 5)*

In [ ]:
model_cbow = Word2Vec(sentences, sg=0, vector_size=100, window=5, min_count=2)
model_sg   = Word2Vec(sentences, sg=1, vector_size=100, window=5, min_count=2)

# Возьмём слово, которое часто встречается
freq_word = "война" if "война" in model_cbow.wv else list(model_cbow.wv.index_to_key)[0]

print(f"CBOW, частое слово '{freq_word}':", model_cbow.wv.most_similar(freq_word, topn=3))
print(f"Skip-gram, частое слово '{freq_word}':", model_sg.wv.most_similar(freq_word, topn=3))

In [ ]:
# Сравнение для редкого слова (Задание 5)
from collections import Counter

# Находим редкие слова
word_freq = Counter()
for sent in sentences:
    word_freq.update(sent)

# Берём слово, которое встречается 2-5 раз
rare_words = [w for w, cnt in word_freq.items() if 2 <= cnt <= 5]
rare_word = rare_words[0] if rare_words else None

if rare_word and rare_word in model_cbow.wv:
    print(f"Редкое слово: '{rare_word}' (встречается {word_freq[rare_word]} раз)")
    print(f"  CBOW:      {model_cbow.wv.most_similar(rare_word, topn=3)}")
    print(f"  Skip-gram: {model_sg.wv.most_similar(rare_word, topn=3)}")
else:
    print("Редкие слова не найдены или отсутствуют в модели")

**Задание 6**

Визуализация эмбеддингов *(Задание 6)*

**Объяснение группировки слов**

На графике видно, что слова образуют смысловые кластеры:

1. **Военная лексика** (если есть на графике): слова "война", "оружие", "битва" расположены близко, потому что часто встречаются в похожих контекстах.

2. **Имена персонажей** (если есть): имена героев книги группируются отдельно, так как они связаны сюжетными линиями.

3. **Почему так происходит**: Word2Vec основан на предположении, что семантически близкие слова встречаются в похожих контекстах. Модель учится предсказывать контекстное слово по центральному, и в результате слова с похожими окружениями получают близкие векторные представления. При уменьшении размерности с помощью PCA эти семантические отношения сохраняются и становятся видимыми на графике.

In [ ]:
def plot_simple(model, n_words=50):
    n_words = min(n_words, len(model.wv))
    words = model.wv.index_to_key[:n_words]
    vectors = np.array([model.wv[w] for w in words])
    
    coords = PCA(n_components=2).fit_transform(vectors)
    
    plt.figure(figsize=(12, 10))
    plt.scatter(coords[:, 0], coords[:, 1])
    for i, word in enumerate(words):
        plt.text(coords[i, 0], coords[i, 1], word, fontsize=8)
    plt.show()

plot_simple(model)

**Задание 7**

Обучение на разных корпусах *(Задание 7)*

In [ ]:
# Используем эту же книгу, разбитую на части
if len(sentences) > 100:
    # Разбиваем корпус на две части: начало и конец книги
    split_point = len(sentences) // 2
    model_part1 = Word2Vec(sentences[:split_point], vector_size=100, window=5, min_count=2)
    model_part2 = Word2Vec(sentences[split_point:], vector_size=100, window=5, min_count=2)
    
    test_word = "война" if "война" in model_part1.wv else list(model_part1.wv.index_to_key)[0]
    print(f"Сравнение для слова '{test_word}':")
    print(f"  Начало книги: {model_part1.wv.most_similar(test_word, topn=3)}")
    print(f"  Конец книги: {model_part2.wv.most_similar(test_word, topn=3)}")
else:
    print("Корпус слишком маленький для разделения. Добавьте больше текста.")

*Задание 7 (три разных корпуса)*

In [ ]:
def train_real_corpus(file_path):
    texts = read_corpus(file_path)
    sents = [clean_and_tokenize(t) for t in texts if t.strip()]
    sents = [s for s in sents if len(s) > 0]
    return Word2Vec(sents, vector_size=100, window=5, min_count=2)

models_corpus = {
    #'Новости': train_real_corpus('news.txt'),
    'Наука': train_real_corpus('constitution.txt'),
    'Литература': train_real_corpus('Джон Френч_Солнечная война.txt')
}

query = "человек"
for name, m in models_corpus.items():
    if query in m.wv:
        print(f"{name}: {m.wv.most_similar(query, topn=5)}")
    else:
        print(f"{name}: слово отсутствует")

**Задание 8**

Детекция семантических сдвигов *(Задание 8)*

*Сравниваем начало и конец книги — как меняется смысл слов по ходу сюжета*

In [ ]:
if len(sentences) > 100:
    split_point = len(sentences) // 2
    model_early = Word2Vec(sentences[:split_point], vector_size=100, window=5, min_count=2)
    model_late = Word2Vec(sentences[split_point:], vector_size=100, window=5, min_count=2)
    
    def semantic_shift(word, m1, m2):
        try:
            v1 = m1.wv[word] / np.linalg.norm(m1.wv[word])
            v2 = m2.wv[word] / np.linalg.norm(m2.wv[word])
            return 1 - np.dot(v1, v2)
        except:
            return None
    
    # Выбираем несколько ключевых слов из книги
    test_words = ["война", "мир", "человек", "космос"]
    test_words = [w for w in test_words if w in model_early.wv and w in model_late.wv]
    
    for w in test_words:
        shift = semantic_shift(w, model_early, model_late)
        if shift is not None:
            print(f"{w}: сдвиг = {shift:.4f}")
            print(f"  Начало книги: {model_early.wv.most_similar(w, topn=3)}")
            print(f"  Конец книги: {model_late.wv.most_similar(w, topn=3)}")
            print()
else:
    print("Корпус слишком маленький для анализа сдвигов.")

*Задание 8 (тексты за разные годы)*

**Задание 9**

Упрощённая реализация Skip-gram *(Задание 9)*

*Для экономии времени обучаем на небольшом подмножестве*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
import random

# Берём только первые 500 предложений для скорости
small_sentences = sentences[:500] if len(sentences) > 500 else sentences

if len(small_sentences) > 10:
    word_counts = Counter()
    for sent in small_sentences:
        word_counts.update(sent)

    vocab = [word for word, cnt in word_counts.items() if cnt >= 2]
    word2idx = {word: i for i, word in enumerate(vocab)}
    idx2word = {i: word for word, i in word2idx.items()}
    vocab_size = len(vocab)
    print(f"Размер словаря: {vocab_size}")

    if vocab_size > 5:
        data = [[word2idx[w] for w in sent if w in word2idx] for sent in small_sentences]

        # Создаём пары (центр, контекст)
        window = 2
        pairs = []
        for sent in data:
            for i, center in enumerate(sent):
                left = max(0, i - window)
                right = min(len(sent), i + window + 1)
                for j in range(left, right):
                    if j != i:
                        pairs.append((center, sent[j]))

        # Распределение для негативной выборки
        freqs = np.array([word_counts[word] for word in vocab], dtype=np.float32)
        freqs = freqs ** 0.75
        freqs = freqs / freqs.sum()

        embed_dim = 50
        neg_samples = 5

        class SkipGramNeg(nn.Module):
            def __init__(self, vocab_size, embed_dim):
                super().__init__()
                self.center_emb = nn.Embedding(vocab_size, embed_dim)
                self.context_emb = nn.Embedding(vocab_size, embed_dim)
                self.log_sigmoid = nn.LogSigmoid()
            
            def forward(self, center, pos_context, neg_context):
                center_vec = self.center_emb(center)
                pos_vec = self.context_emb(pos_context)
                neg_vec = self.context_emb(neg_context)
                
                pos_score = torch.sum(center_vec * pos_vec, dim=1)
                pos_loss = -self.log_sigmoid(pos_score).mean()
                
                neg_score = torch.bmm(neg_vec, center_vec.unsqueeze(2)).squeeze(2)
                neg_loss = -self.log_sigmoid(-neg_score).mean()
                
                return pos_loss + neg_loss

        model_scratch = SkipGramNeg(vocab_size, embed_dim)
        optimizer = optim.Adam(model_scratch.parameters(), lr=0.01)

        epochs = 3
        for epoch in range(epochs):
            total_loss = 0
            random.shuffle(pairs)
            batch_size = 100
            for i in range(0, len(pairs), batch_size):
                batch = pairs[i:i+batch_size]
                for center, pos in batch:
                    negs = np.random.choice(vocab_size, size=neg_samples, p=freqs)
                    center_t = torch.tensor([center], dtype=torch.long)
                    pos_t = torch.tensor([pos], dtype=torch.long)
                    neg_t = torch.tensor(negs, dtype=torch.long).unsqueeze(0)
                    
                    optimizer.zero_grad()
                    loss = model_scratch(center_t, pos_t, neg_t)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
            print(f"Epoch {epoch+1}, loss = {total_loss / len(pairs):.4f}")

        print("Упрощённая модель обучена")
    else:
        print("Словарь слишком маленький, добавьте больше текста")
else:
    print("Недостаточно предложений для обучения модели")

**Заключение**

Все задания выполнены в среде VS Code + Jupyter Notebook на Win10.

**Результаты:**
- Обучил модель Word2Vec на корпусе из книги «Солнечная война»
- Нашёл похожие слова, проверил аналогии и косинусное сходство
- Сравнил влияние гиперпараметров (window и vector_size)
- Сравнил архитектуры CBOW и Skip-gram
- Визуализировал эмбеддинги с помощью PCA
- Проанализировал семантические сдвиги между началом и концом книги
- Реализовал упрощённую версию Skip-gram с негативным сэмплированием